## Modeling - Evaluation
## CRISP-DM – Fase 4 e 5
---

In [2]:
# Importação das libraries

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from pathlib import Path


# Paths do projeto

BASE_DIR = Path().resolve()

DATA_DIR = BASE_DIR / "data"
DATA_PREPARED_DIR = DATA_DIR / "prepared"

OUTPUTS_DIR = BASE_DIR / "outputs"
FIGURES_BASE_DIR = OUTPUTS_DIR / "figures"

FIGURES_UNDERSTANDING_DIR = FIGURES_BASE_DIR / "understanding"
FIGURES_PREPARATION_DIR = FIGURES_BASE_DIR / "preparation"
FIGURES_MODELING_DIR = FIGURES_BASE_DIR / "modeling"

# Criar pastas automaticamente
DATA_PREPARED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_UNDERSTANDING_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_PREPARATION_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_MODELING_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 1. Carregamento dos dados (X = perguntas/features, y = resposta/target)
X_train = pd.read_csv(DATA_PREPARED_DIR / "X_train_scaled.csv")
X_test = pd.read_csv(DATA_PREPARED_DIR / "X_test_scaled.csv")

# Usamos o squeeze para transformar a tabela de uma coluna num vetor simples (1D)
y_train = pd.read_csv(DATA_PREPARED_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(DATA_PREPARED_DIR / "y_test.csv").squeeze()

# 2. Confirmar se os dados estão corretos
print("--- Check de Dados ---")
print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")

# 3. Criar a Baseline (Ponto de comparação)
# Previsão constante usando a média do treino
y_mean_val = y_train.mean()
y_pred_baseline = [y_mean_val] * len(y_test)

# Cálculo de todas as métricas para a Baseline
mae_base = mean_absolute_error(y_test, y_pred_baseline)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
r2_base = r2_score(y_test, y_pred_baseline) # Por definição será 0.0

print(f"--- Baseline (Média) ---")
print(f"Média Calculada: {y_mean_val:.2f}€")
print(f"MAE a bater: {mae_base:.2f}€")

--- Check de Dados ---
Treino: (160, 12) | Teste: (40, 12)
--- Baseline (Média) ---
Média Calculada: 1421.65€
MAE a bater: 261.69€


In [16]:
# 1. Definição dos Modelos
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42, n_estimators=200
                                           )
}

# 2. Dicionário para resultados
resultados = {}

# 3. Adicionar a Baseline (Média) manualmente ao dicionário primeiro
# Isso garante que seja a primeira linha da nossa tabela comparativa
resultados["Baseline (Média)"] = {
    'MAE': mae_base, 
    'RMSE': rmse_base, 
    'R²': r2_base
}

# 4. Treinar e Avaliar os modelos em loop
for nome, modelo in models.items():
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    
    # Cálculo das métricas
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    resultados[nome] = {'MAE': mae, 'RMSE': rmse, 'R²': r2}

# 5. Exibição da Tabela Final
df_resultados = pd.DataFrame(resultados).T
display(df_resultados.round(4))

,MAE,RMSE,R²
Baseline (Média),261.6850,318.9182,-0.0022
Linear Regression,277.0610,338.4978,-0.1291
Decision Tree,440.7500,514.7475,-1.6109
Random Forest,291.4329,355.1487,-0.2429


<hr>

# **Síntese: Modeling e Evaluation**

A convergência de resultados entre os modelos testados confirma a ausência de sinal preditivo no dataset. A estrutura de dados atual não permite modelar a variável `Salary` com fiabilidade estatística.

### **1. Validação por Baseline (Média)**
O rigor metodológico foi assegurado pela comparação dos modelos com uma **Baseline Determinística** (média do treino):
* **Superioridade da Baseline:** O menor erro foi obtido pela média (**MAE: 261.68€**). Nenhum algoritmo de Machine Learning superou esta métrica, indicando que o erro residual é de natureza puramente aleatória.
* **Coeficientes de Determinação ($R^2$):** A obtenção de valores negativos (entre **-0.12** e **-1.61**) ratifica que os modelos introduzem mais ruído do que uma previsão constante, falhando em explicar a variabilidade dos dados.

### **2. Comportamento Algorítmico**
* **Regressão Linear:** Revelou-se o modelo mais estável pela sua simplicidade, embora sem utilidade prática.
* **Modelos Não-Lineares:** A **Decision Tree** apresentou um sobreajustamento (*overfitting*) severo ($R^2 = -1.61$), mapeando padrões falsos que não generalizam para o conjunto de teste.

### **3. Porque descartamos Feature Engineering?**

A aplicação de técnicas de Feature Engineering (como rácios ou combinações polinomiais) foi considerada, mas descartada com base em dois pilares técnicos:

- **Propagação do Ruído:** O Feature Engineering atua como um amplificador de sinais existentes. Quando as variáveis base apresentam correlação nula com o target, qualquer transformação matemática resultará apenas na propagação e complexificação do ruído original. Matematicamente, não é possível extrair informação preditiva onde não existe variância partilhada.

- **Sinal Fraco vs. Ausência de Sinal:** Ao contrário de cenários com correlações baixas mas persistentes (ex: $0.15$ a $0.25$) — onde interações poderiam expor relações não-lineares — os dados atuais situam-se abaixo do limiar de significância. Nestas condições, criar novas colunas apenas aumentaria a dimensionalidade sem ganho real, elevando o risco de correlações falsas.

<br>
<hr>

## **Conclusões de Negócio**

1.  **Assimetria nos Determinantes Salariais:** Não existe evidência de causalidade entre as métricas de mérito (Avaliação), compromisso (Assiduidade) ou perfil académico e a remuneração.
2.  **Hipótese de Variáveis Exógenas:** A variância salarial é provavelmente determinada por fatores não capturados (ex.: negociação individual, subjetividade departamental ou variáveis de mercado).
3.  **Inviabilidade Preditiva:** Sob as condições atuais, a implementação de um modelo preditivo carece de viabilidade técnica. Recomenda-se a revisão das métricas de entrada ou a inclusão de dados históricos externos.